# PF400 Simulation Node Tests

Exercises every action of the PF400 node running in **simulation mode**: no telnet
connection, no arm movement, but the full pre-flight and the resource bookkeeping run.

The notebook starts the node itself, so the only prerequisite is a reachable MADSci
resource manager.

**Do not point this at the resource manager a real lab is using.** A simulation node
writes resource changes exactly as a real one does. That is deliberate, it is what lets
a multi-step workflow be checked, but it means the writes land wherever you point it.


In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
SRC = REPO / "src"

NODE_PORT = 2010
NODE_URL = f"http://localhost:{NODE_PORT}"
RESOURCE_SERVER_URL = os.environ.get("RESOURCE_SERVER_URL", "http://localhost:8003")

print("repo           ", REPO)
print("node will serve", NODE_URL)
print("resource server", RESOURCE_SERVER_URL)

## 1. Start the node

Simulation mode is a startup flag, not a per-request one. A node that never opened a
connection cannot move the arm whatever state it is in.

Note the environment variable names. The prefix is `NODE_`, so `simulation` becomes
`NODE_SIMULATION`. Fields whose own name already begins with `node_` are not doubled:
`node_url` is `NODE_URL`, not `NODE_NODE_URL`. Getting this wrong fails quietly, the
node just keeps its default port and collides with whatever already has it.


In [ ]:
env = {
    **os.environ,
    "PYTHONPATH": str(SRC),
    "RESOURCE_SERVER_URL": RESOURCE_SERVER_URL,
    "NODE_SIMULATION": "true",
    "NODE_NAME": "pf400_sim_notebook",
    "NODE_URL": f"http://0.0.0.0:{NODE_PORT}",
}

log = open("/tmp/pf400_sim_notebook.log", "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "pf400_rest_node"],
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT,
)
print("started pid", proc.pid)

In [ ]:
def wait_ready(url, timeout=90):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            urllib.request.urlopen(url + "/status", timeout=3)
            return True
        except Exception:
            time.sleep(1)
    return False


assert wait_ready(NODE_URL), "node did not come up, see /tmp/pf400_sim_notebook.log"
print("node is up")

In [ ]:
from madsci.client.node.rest_node_client import RestNodeClient
from madsci.common.types.action_types import ActionRequest
from madsci.common.types.location_types import LocationArgument

client = RestNodeClient(url=NODE_URL)
state = client.get_state()
print(state)
assert state["pf400_status_code"] == "SIMULATION", "node is not in simulation mode"
assert state["simulation"] is True

## 2. Test helper

Every test states the outcome it expects. A test that passes for the wrong reason is
worse than one that fails, so the refusal text is printed in full.


In [ ]:
results = []


def run(name, action_name, args, expect):
    """Send an action and compare the outcome against what the test expects."""
    req = ActionRequest(action_name=action_name, args=args)
    res = client.send_action(req)
    status = str(getattr(res.status, "value", res.status))
    ok = status == expect
    results.append((name, ok))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    print(f"       expected {expect}, got {status}")
    for e in res.errors or []:
        print("       reason:", getattr(e, "message", e))
    print()
    return res


def loc(name, joints, rotation="narrow", resource_id=None, **extra):
    rep = {"location": joints, "plate_rotation": rotation, **extra}
    return LocationArgument(
        location_name=name, representation=rep, resource_id=resource_id
    ).model_dump(mode="json")

## 3. Test locations

Real joint values taken from the RAPID 446 location manager, so the poses are ones the
arm genuinely reaches.


In [ ]:
SAFE_HOME = [989.443, -2.007, 174.876, 542.366, 70.574, -843.897]
EXCHANGE_HIGH = [638.532, -19.079, 65.561, 732.286, 78.587, 752.82]
EXCHANGE_LOW = [632.616, -19.079, 65.561, 732.286, 78.587, 752.82]

# Deliberately bad poses for the negative tests.
Z_TOO_HIGH = [1200.0, 10.0, 120.0, 0.0, 90.0, 100.0]  # past the 1161.5 soft stop
Z_NEAR_TOP = [1155.0, 10.0, 120.0, 0.0, 90.0, 100.0]  # legal until +15 approach
RAIL_TOO_FAR = [600.0, 10.0, 120.0, 0.0, 90.0, 1500.0]  # past the 1000 rail limit

## 4. transfer

Including the three refusals a planning agent most needs to see.


In [ ]:
run(
    "transfer, valid",
    "transfer",
    {
        "source": loc("exchange_high", EXCHANGE_HIGH),
        "target": loc("exchange_low", EXCHANGE_LOW),
    },
    "succeeded",
)

In [ ]:
run(
    "transfer, rotation changes with no rotation_deck",
    "transfer",
    {
        "source": loc("exchange_high", EXCHANGE_HIGH, "wide"),
        "target": loc("exchange_low", EXCHANGE_LOW, "narrow"),
    },
    "failed",
)

In [ ]:
run(
    "transfer, target past the z soft limit",
    "transfer",
    {
        "source": loc("exchange_high", EXCHANGE_HIGH),
        "target": loc("too_high", Z_TOO_HIGH),
    },
    "failed",
)

In [ ]:
# z is legal at 1155 but the approach adds 15, which is not.
run(
    "transfer, approach position past the z soft limit",
    "transfer",
    {
        "source": loc("exchange_high", EXCHANGE_HIGH),
        "target": loc("near_top", Z_NEAR_TOP),
    },
    "failed",
)

In [ ]:
run(
    "transfer, rail past its soft limit",
    "transfer",
    {
        "source": loc("exchange_high", EXCHANGE_HIGH),
        "target": loc("rail_far", RAIL_TOO_FAR),
    },
    "failed",
)

In [ ]:
run(
    "transfer, invalid plate_rotation value",
    "transfer",
    {
        "source": loc("exchange_high", EXCHANGE_HIGH, "sideways"),
        "target": loc("exchange_low", EXCHANGE_LOW),
    },
    "failed",
)

## 5. pick_plate and place_plate

These need real resources, so the occupancy checks have something to read.


In [ ]:
from madsci.client.resource_client import ResourceClient
from madsci.common.types.resource_types import Asset, Slot

rc = ResourceClient(RESOURCE_SERVER_URL)
TAG = f"nbtest_{int(time.time())}"

nests = {
    n: rc.add_resource(
        Slot(resource_name=f"{TAG}_{n}", resource_class="NotebookNest", capacity=1)
    )
    for n in ("A", "B", "C")
}
plate = rc.add_resource(
    Asset(resource_name=f"{TAG}_plate", resource_class="NotebookPlate")
)
rc.push(resource=nests["A"].resource_id, child=plate)


def occupancy():
    return {k: rc.get_resource(v.resource_id).quantity for k, v in nests.items()}


print("created", TAG)
print("occupancy", occupancy())

In [ ]:
run(
    "pick_plate from an occupied nest",
    "pick_plate",
    {"source": loc("nest_A", EXCHANGE_HIGH, resource_id=nests["A"].resource_id)},
    "succeeded",
)
print("occupancy", occupancy())

In [ ]:
run(
    "place_plate into an empty nest",
    "place_plate",
    {"target": loc("nest_B", EXCHANGE_LOW, resource_id=nests["B"].resource_id)},
    "succeeded",
)
print("occupancy", occupancy())

In [ ]:
run(
    "pick_plate from a nest that is now empty",
    "pick_plate",
    {"source": loc("nest_A", EXCHANGE_HIGH, resource_id=nests["A"].resource_id)},
    "failed",
)

In [ ]:
run(
    "place_plate into a nest that is already occupied",
    "place_plate",
    {"target": loc("nest_B", EXCHANGE_LOW, resource_id=nests["B"].resource_id)},
    "failed",
)

## 6. move_to_location and move_neutral


In [ ]:
run(
    "move_to_location, reachable",
    "move_to_location",
    {"target": loc("safe_home", SAFE_HOME)},
    "succeeded",
)

In [ ]:
run(
    "move_to_location, unreachable",
    "move_to_location",
    {"target": loc("too_high", Z_TOO_HIGH)},
    "failed",
)

In [ ]:
run("move_neutral", "move_neutral", {}, "succeeded")

## 7. remove_lid and replace_lid

Both delegate to `transfer` in the driver, so they inherit its checks.


In [ ]:
run(
    "remove_lid",
    "remove_lid",
    {
        "source": loc("exchange_high", EXCHANGE_HIGH),
        "target": loc("exchange_low", EXCHANGE_LOW),
    },
    "succeeded",
)

In [ ]:
run(
    "replace_lid",
    "replace_lid",
    {
        "source": loc("exchange_low", EXCHANGE_LOW),
        "target": loc("exchange_high", EXCHANGE_HIGH),
    },
    "succeeded",
)

In [ ]:
run(
    "remove_lid, target unreachable",
    "remove_lid",
    {
        "source": loc("exchange_high", EXCHANGE_HIGH),
        "target": loc("too_high", Z_TOO_HIGH),
    },
    "failed",
)

## 8. A multi-step sequence

This is the reason a simulation node writes resources rather than refusing to.
Step 2 can only be checked if step 1's effect is visible. Validating against a
resource graph that never changes lets you check the first step and nothing after it.


In [ ]:
print("start     ", occupancy())

run(
    "sequence step 1, B to C",
    "transfer",
    {
        "source": loc("nest_B", EXCHANGE_HIGH, resource_id=nests["B"].resource_id),
        "target": loc("nest_C", EXCHANGE_LOW, resource_id=nests["C"].resource_id),
    },
    "succeeded",
)
print("after 1   ", occupancy())

run(
    "sequence step 2, C to A, depends on step 1",
    "transfer",
    {
        "source": loc("nest_C", EXCHANGE_LOW, resource_id=nests["C"].resource_id),
        "target": loc("nest_A", EXCHANGE_HIGH, resource_id=nests["A"].resource_id),
    },
    "succeeded",
)
print("after 2   ", occupancy())

run(
    "sequence step 3, B to C again, source is empty now",
    "transfer",
    {
        "source": loc("nest_B", EXCHANGE_HIGH, resource_id=nests["B"].resource_id),
        "target": loc("nest_C", EXCHANGE_LOW, resource_id=nests["C"].resource_id),
    },
    "failed",
)

## 9. Results


In [ ]:
passed = sum(1 for _, ok in results if ok)
print(f"{passed}/{len(results)} passed")
for name, ok in results:
    if not ok:
        print("  FAILED:", name)
assert passed == len(results), "some tests failed"

## 10. Clean up


In [ ]:
# Remove the plate before the nests. Deleting a nest cascades to whatever it
# holds, so clearing the nests first leaves this looking for a plate that is
# already gone and reporting a 404 that is not a real failure.
for res in [plate, *nests.values()]:
    try:
        rc.remove_resource(res.resource_id)
    except Exception:
        print("already removed:", res.resource_name)

proc.terminate()
proc.wait(timeout=20)
log.close()
print("node stopped, test resources removed")